# Calculating T90 for Mask Weighted counts

In [1]:
import batanalysis as ba
import swiftbat
from swifttools.swift_too import ObsQuery 
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from pathlib import Path
from astropy.io import fits
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import SkyCoord
import datetime
import os
from swifttools.swift_too import GUANO
from swifttools.swift_too import Data
from astropy.stats import bayesian_blocks  

In [13]:
def calculate_t90_grb_mask_weighted(
    trigger_time:float,
    event
):
     # Helper function to compute cumulative counts over time
    def tc(t1,edges,rates):
        rate_t=np.zeros_like(t1)* (u.ct / u.s)
        for i in range(len(t1)):
            ind=np.digitize(t1[i],edges)-1
            if ind < 0:
                ind = 0
            elif ind >= len(rates):
                ind = len(rates) - 1
            rate_t[i] = rates[ind]
        dt = t1[1]-t1[0]
        int_t=np.cumsum(rate_t)*dt
        return int_t
    
    # Step 1: Create a uniform (1s bin) mask-weighted lightcurve
    lc = event.create_lightcurve(timedelta=np.timedelta64(1, "s"), recalc = True)
    
    # Step 2: Extract time, rate, and error for energy band index 4
    t = lc.data['TIME']
    x = lc.data['RATE'][:,4]
    sigma = lc.data['ERROR'][:,4]
    
    # Step 3: Use Bayesian blocks to adaptively segment the lightcurve
    edges = bayesian_blocks(t, x, sigma, fitness='measures')
    
    # Step 4: Re-create lightcurve using the Bayesian block edges
    lgc = event.create_lightcurve(timebins=edges*u.s, recalc = 'True')
    
    # Step 5: Get rates and edges, excluding the first and last block
    rates = lgc.data['RATE'][1:-1,4]
    edges_needed = edges[1:-1]
    
    # Step 6: Set up a fine time array for interpolation (0.01s resolution)
    t_min = edges_needed[0]
    t_max = edges_needed[-1]
    t_step = 0.01
    t1 = np.arange(t_min, t_max+t_step, t_step)
    
    # Step 7: Compute cumulative counts over time
    cumulative_counts = tc(t1, edges_needed, rates)
    total_counts = cumulative_counts[-1]
    
    # Step 8: Compute T90 start, end, and duration (relative to trigger time)
    lower_5 = 0.05 * total_counts
    upper_95 = 0.95 * total_counts
    idx_5 = np.searchsorted(cumulative_counts, lower_5)
    idx_95 = np.searchsorted(cumulative_counts, upper_95)
    t_start = t1[idx_5]-trigger_time
    t_end   = t1[idx_95]-trigger_time
    T90 = t_end - t_start
    return t_start, t_end, T90

# Example

In [8]:
oq = ObsQuery(begin="2024-04-21 09:43:24", length=0.1)#Collecting the data
oq

2024-04-21 09:28:35,2024-04-21 09:43:37,CXOU J164710.2-455216,00015870016,775,127
2024-04-21 09:43:42,2024-04-21 10:14:28,,01223470000,1740,106
2024-04-21 10:14:29,2024-04-21 10:15:41,4U 1909+07,00016436027,0,72
2024-04-21 10:15:44,2024-04-21 10:20:58,GRB 240419B,01223072006,225,89
2024-04-21 10:21:02,2024-04-21 10:26:57,J231617.6-135545,00090964002,270,85
2024-04-21 10:27:02,2024-04-21 10:44:00,AT2019QIZ,00097575004,820,198
2024-04-21 10:44:02,2024-04-21 10:53:57,AT2024as,00016500014,445,150
2024-04-21 10:54:02,2024-04-21 11:06:59,NGC 4395,00097137183,675,102
2024-04-21 11:07:04,2024-04-21 11:15:58,UCAC4 744-049386,00097340001,450,84
2024-04-21 11:16:02,2024-04-21 11:35:57,,01223470001,1025,170
2024-04-21 11:36:02,2024-04-21 11:38:26,T CRB,00097564012,0,144


In [9]:
data = Data(obsid=oq[1].obsid, bat=True, outdir="~/Downloads/", clobber=True)#Downloading the data

In [10]:
event=ba.BatEvent(oq[1].obsid,obs_dir = "/storage/home/zbl5364/work/Downloads/", is_guano=True, recalc = True, ra = 299.68*u.deg, dec = -14.87*u.deg)

A save file has been written to /storage/work/zbl5364/Downloads/01223470000_eventresult/batevent.pickle.


In [14]:
t_start, t_end, T90 = calculate_t90_grb_mask_weighted(
    trigger_time=735385407.36,
    event=event
)

print(f"T90 start = {t_start:.3f}")
print(f"T90 end   = {t_end:.3f}")
print(f"T90       = {T90:.3f}")

T90 start = -15.780
T90 end   = 5.150
T90       = 20.930
